# AI 생성 기사 이진 분류 — KLUE-RoBERTa 모델 예측

**목적**: 낚시성 패턴을 의도적으로 적용해 생성한 기사와 정상 기사를 모델에 입력하여, 모델이 패턴을 제대로 학습했는지 확인  
**모델**: `klue_binary_final.pt` (전체 데이터 재학습본)  
**구성**: 낚시성 의도 5건 + 정상 의도 5건 = 총 10건  
**방식**: 정답 라벨 없음 — 모델이 판단한 결과와 확률만 출력

In [ ]:
import os, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_DIR   = os.getcwd()
MODEL_NAME = 'klue/roberta-base'
BIN_PT     = os.path.join(BASE_DIR, 'klue_binary_final.pt')
MAX_LEN    = 128
LABEL      = {0: '정상', 1: '낚시성'}

print('설정 완료')
print(f'모델 파일 존재: {os.path.exists(BIN_PT)}')

In [ ]:
print('토크나이저 및 모델 로딩 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.load_state_dict(torch.load(BIN_PT, map_location='cpu'))
model.eval()
print('로딩 완료!')

In [ ]:
# AI가 생성한 기사 10건 (제목, 본문 앞부분, 의도 패턴)
# 패턴 설명:
#   뭐길래      : 핵심 정보를 숨기고 궁금증 유발
#   알고보니    : 반전/충격 암시
#   경고/충격   : 선정적 표현으로 위기감 조성
#   터졌다      : 사건 과장 표현
#   알고보니+비법 : 비법 암시로 클릭 유도

GENERATED = [
    # ── 낚시성 의도 5건 ───────────────────────────────────────────────
    ('패딩 하나에 300만원…요즘 MZ가 줄 서는 브랜드 뭐길래',
     '최근 20~30대 사이에서 고가 아웃도어 브랜드에 대한 수요가 폭발적으로 증가하고 있다. '
     '일부 제품은 출시 당일 품절되는 현상이 반복되고 있으며, 중고 시장에서는 정가의 두 배 이상에 거래되는 사례도 나타나고 있다. '
     '전문가들은 희소성 마케팅과 SNS 확산이 주요 요인이라고 분석한다.',
     '뭐길래 패턴'),

    ('20대 직장인, 알고보니 연봉 10억…"회사 이름은 말 못해"',
     '평범한 20대 직장인으로 알려진 A씨가 실제로는 연봉 10억 원을 받는 것으로 알려져 화제다. '
     'A씨는 SNS에 평범한 일상을 공유해 왔으나, 최근 한 방송에 출연해 고액 연봉 사실을 공개했다. '
     '다만 A씨는 근무하는 회사명은 밝힐 수 없다며 말을 아꼈다.',
     '알고보니 패턴'),

    ('"이 음식 매일 먹으면 큰일 납니다"…의사들이 경고한 충격 식품',
     '건강 전문가들이 특정 가공식품을 매일 섭취할 경우 심각한 건강 문제로 이어질 수 있다고 경고했다. '
     '해당 식품에는 과도한 나트륨과 인공첨가물이 포함돼 있으며, 장기 섭취 시 신장 기능 저하 및 고혈압 위험이 높아진다는 연구 결과가 발표됐다. '
     '의료계에서는 섭취 빈도를 줄이고 천연 식재료로 대체할 것을 권고하고 있다.',
     '경고/충격 패턴'),

    ('아이돌 A, 결국 터졌다…소속사 "사실 확인 중"',
     '최근 온라인 커뮤니티를 중심으로 인기 아이돌 A의 사생활 논란이 불거졌다. '
     '한 누리꾼이 A의 열애설을 제기하는 게시물을 올리면서 논란이 시작됐고, '
     '소속사 측은 현재 사실 확인 중이라는 입장을 밝혔다.',
     '터졌다 패턴'),

    ('"월급 300만원인데 강남 아파트 샀다"…그 방법 알고보니',
     '직장인 B씨는 월 300만 원의 급여로 강남권 아파트를 마련했다고 밝혀 화제를 모으고 있다. '
     'B씨에 따르면 10년간 극도로 절약한 생활비와 부모님의 일부 지원, 전세 레버리지를 활용한 것이 핵심이었다. '
     '부동산 전문가들은 이 같은 방식이 현실적으로 가능하지만 리스크도 크다고 조언했다.',
     '알고보니+비법 패턴'),

    # ── 정상 의도 5건 ─────────────────────────────────────────────────
    ('통계청, 1분기 GDP 성장률 0.8% 발표…내수 회복 신호',
     '통계청이 발표한 올해 1분기 실질 국내총생산(GDP) 성장률이 전 분기 대비 0.8%로 집계됐다. '
     '민간소비가 0.5% 증가하며 내수 회복세를 이끌었고, 수출도 반도체 호황에 힘입어 2.1% 늘었다. '
     '정부는 하반기 경기 개선 흐름이 이어질 것으로 전망했다.',
     '경제 통계 보도'),

    ('서울시, 내년부터 지하철 노인 무임승차 연령 65세→70세 조정 검토',
     '서울시가 현행 만 65세 이상에게 적용되는 지하철 무임승차 연령 기준을 만 70세로 높이는 방안을 검토 중이라고 밝혔다. '
     '시는 고령화 가속으로 연간 무임 손실이 6000억 원을 넘어서면서 재정 부담이 가중되고 있다고 설명했다. '
     '관련 조례 개정을 위한 시의회 논의가 하반기 중 진행될 예정이다.',
     '정책 보도'),

    ('한국 축구대표팀, 월드컵 최종예선 첫 경기 2-1 승리',
     '한국 축구대표팀이 2026 FIFA 북중미 월드컵 아시아 최종예선 첫 경기에서 상대팀을 2-1로 꺾고 승점 3점을 확보했다. '
     '전반 23분 손흥민의 선제골에 이어 후반 11분 황희찬이 추가골을 터뜨렸다. '
     '대표팀 감독은 경기 후 선수들의 집중력이 좋았다고 평가했다.',
     '스포츠 결과 보도'),

    ('국립암센터, 국내 대장암 발생률 5년 연속 감소 추세 확인',
     '국립암센터가 발표한 2024년 암 통계에 따르면 국내 대장암 발생률이 5년 연속 감소한 것으로 나타났다. '
     '이는 대장내시경 검진 수검률 증가와 식이 패턴 개선이 복합적으로 작용한 결과로 분석된다. '
     '다만 50세 미만 젊은 층의 대장암 발생은 소폭 증가해 주의가 필요하다고 연구진은 밝혔다.',
     '의학 연구 보도'),

    ('환경부, 2030년까지 전국 플라스틱 재활용률 70% 목표 발표',
     '환경부가 오는 2030년까지 국내 플라스틱 폐기물 재활용률을 현재 40%에서 70%로 끌어올리겠다는 목표를 발표했다. '
     '이를 위해 생산자 책임 재활용 제도를 강화하고, 재활용 선별 시설 확충에 2000억 원을 투입할 계획이다. '
     '시민단체들은 목표 설정을 환영하면서도 실행 계획의 구체성이 부족하다고 지적했다.',
     '환경 정책 보도'),
]

print(f'생성 기사 총 {len(GENERATED)}건 준비 완료')
print('낚시성 의도:', len([g for g in GENERATED if '패턴' in g[2]]), '건')
print('정상 의도  :', len([g for g in GENERATED if '보도' in g[2]]), '건')

In [ ]:
import pandas as pd

rows = []
for title, content, pattern in GENERATED:
    enc = tokenizer(
        text=title,
        text_pair=content,
        truncation='only_second',
        max_length=MAX_LEN,
        padding='max_length',
        return_tensors='pt',
    )
    with torch.no_grad():
        out = model(**enc)
    prob = F.softmax(out.logits, dim=-1).squeeze()
    pred = int(torch.argmax(prob))
    rows.append({
        '제목':        title,
        '의도 패턴':   pattern,
        '모델 판정':   LABEL[pred],
        '정상 확률':   f'{prob[0]*100:.2f}%',
        '낚시성 확률': f'{prob[1]*100:.2f}%',
    })

df = pd.DataFrame(rows)
df.index = range(1, len(df)+1)
df

In [ ]:
clickbait_count = sum(1 for r in rows if r['모델 판정'] == '낚시성')
normal_count    = sum(1 for r in rows if r['모델 판정'] == '정상')

print('=' * 55)
print('  모델 판정 요약')
print('=' * 55)
print(f'  전체 기사    : {len(rows)}건')
print(f'  낚시성 판정  : {clickbait_count}건')
print(f'  정상 판정    : {normal_count}건')
print('=' * 55)

print('\n  낚시성으로 판정된 기사:')
for r in rows:
    if r['모델 판정'] == '낚시성':
        print(f"  - [{r['의도 패턴']}] {r['제목'][:45]}")
        print(f"    낚시성 확률: {r['낚시성 확률']}")

print('\n  정상으로 판정된 기사:')
for r in rows:
    if r['모델 판정'] == '정상':
        print(f"  - [{r['의도 패턴']}] {r['제목'][:45]}")
        print(f"    정상 확률: {r['정상 확률']}")